# Policy Search Assistant v2 — Setup (copied from v1, unchanged)

In [1]:
import os
from google import genai
from dotenv import load_dotenv, find_dotenv
from pypdf import PdfReader
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
client=genai.Client(api_key=api_key)
reader=PdfReader("CFPB Udaap.pdf")
document_text=""
for page in reader.pages:
    document_text+=page.extract_text()


def chunk_text(text, chunk_size=800, overlap=100):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters shared between consecutive chunks (keeps context from being cut mid-sentence)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward, but overlap with previous chunk
    return chunks

chunks = chunk_text(document_text)
print(f"Number of chunks: {len(chunks)}")
print("--- First chunk ---")
print(chunks[0])
print("--- Second chunk ---")
print(chunks[1])
print(len(chunks))

def embed_chunks(chunks, client, model="gemini-embedding-001"):
    """
    Generates an embedding vector for each text chunk using Gemini's embedding model.
    Returns a list of vectors, same order as input chunks.
    """
    embeddings = []
    for chunk in chunks:
        result = client.models.embed_content(
            model=model,
            contents=chunk
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings

embeddings = embed_chunks(chunks, client)
print(f"Number of embeddings: {len(embeddings)}")
print(f"Vector length (dimensions): {len(embeddings[0])}")
print(f"First 5 values of first vector: {embeddings[0][:5]}")

import faiss
import numpy as np

embeddings_np = np.array(embeddings).astype('float32')

# Normalize vectors to unit length — this is what makes inner product == cosine similarity
faiss.normalize_L2(embeddings_np)

dimension = embeddings_np.shape[1]
index = faiss.IndexFlatIP(dimension)  # IP = inner product

index.add(embeddings_np)

print(f"Number of vectors in index: {index.ntotal}")  # should print 63

chunk_id_to_text = {i: chunks[i] for i in range(len(chunks))}

print(f"Number of entries in mapping: {len(chunk_id_to_text)}")
print(f"Sample entry (id 0): {chunk_id_to_text[0][:150]}...")

def search_policy(query, index, chunk_id_to_text, client, model="gemini-embedding-001", top_k=3):
    """
    Embeds a query, searches the FAISS index, and returns the top_k matching chunks.
    """
    # 1. Embed the query the same way chunks were embedded
    result = client.models.embed_content(model=model, contents=query)
    query_vector = np.array([result.embeddings[0].values]).astype('float32')

    # 2. Normalize the query vector — required for cosine-equivalence with IndexFlatIP
    faiss.normalize_L2(query_vector)

    # 3. Search the index
    scores, indices = index.search(query_vector, top_k)

    # 4. Map FAISS index positions back to chunk text
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": int(idx),
            "score": float(score),
            "text": chunk_id_to_text[idx] if isinstance(list(chunk_id_to_text.keys())[0], int) else chunk_id_to_text[str(idx)]
        })
    return results

# Test it
test_query = "What counts as an unfair act under UDAAP?"
results = search_policy(test_query, index, chunk_id_to_text, client)

for r in results:
    print(f"\n--- Chunk {r['chunk_id']} (score: {r['score']:.4f}) ---")
    print(r['text'][:300])



Number of chunks: 63
--- First chunk ---
CFPB Consumer 
Laws and Regulations UDAAP 
CFPB Manual V.2 (October 2012) UDAAP 1 
Unfair, Deceptive, or Abusive Acts or Practices 
Unfair, deceptive, or abusive acts and practices (UDAAPs) can cause significant financial injury to 
consumers, erode consumer confidence, and undermine the financial marketplace. Under the 
Dodd-Frank Act, it is unlawful for any provider of consumer financial products or services or a 
service provider to engage in any unfair, deceptive or abusive act or practice.1 The Act also 
provides CFPB with rule-making authority and, with respect to entities within its jurisdiction, 
enforcement authority to prevent unfair, deceptive, or abusive acts or practices in connection with 
any transaction with a consumer for a consumer financial product or service, or the off
--- Second chunk ---
ection with 
any transaction with a consumer for a consumer financial product or service, or the offering of a 
consumer financial produc

In [2]:
# =============================================================
# 7A.1 EVAL SETUP: QUERIES + RETRIEVAL WRAPPER + KEYWORD FINDER
# What this does:
#   1. Defines the 6 test queries (direct, vague, mismatch, out-of-scope).
#   2. Defines retrieve(), a wrapper around search_policy() so later steps
#      (multi-query, HyDE) can swap in their own retrieval without touching eval code.
#   3. Defines find_chunks(), a keyword search across all chunks, for
#      checking what the document covers.
# Makes no API calls.
# =============================================================

eval_queries = [
    {"id": "q1", "type": "direct",       "query": "What counts as an unfair act under UDAAP?"},
    {"id": "q2", "type": "direct",       "query": "How does UDAAP define an abusive practice?"},
    {"id": "q3", "type": "vague",        "query": "Can we hide fees in the fine print?"},
    {"id": "q4", "type": "vague",        "query": "Is it okay to pressure customers into signing up quickly?"},
    {"id": "q5", "type": "mismatch",     "query": "What makes a disclosure misleading to a reasonable consumer?"},
    {"id": "q6", "type": "out_of_scope", "query": "What is the minimum credit score for a mortgage?"},
]

K = 5  # how many chunks to retrieve per query

def retrieve(query, k=K):
    return search_policy(query, index, chunk_id_to_text, client, top_k=k)

def find_chunks(keyword):
    return [i for i, t in chunk_id_to_text.items() if keyword.lower() in t.lower()]

In [3]:
# # =============================================================
# # 7A.1 LABELING HELPER: READ CANDIDATE CHUNKS + KEYWORD CHECKS
# # What this does:
# #   1. Prints the full text of the candidate chunks (picked from the
# #      baseline top-5 output), so you can decide by reading which ones
# #      actually contain the answer to each eval query.
# #   2. Keyword-searches the whole document for terms related to q4
# #      ("pressure", "urgent", etc.) to check whether the document covers
# #      that topic directly at all.
# # Output: reading material for labeling. It doesn't compute any metrics
# # and nothing from it is saved.
# # =============================================================

# # Chunk IDs to read in full (from the baseline top-5s plus their neighbors)
# candidates = [3, 4, 5, 33, 34, 35, 18, 19, 21, 22, 27, 28, 53]
# for cid in candidates:
#     print(f"\n===== chunk {cid} =====")
#     print(chunk_id_to_text[cid])  # full chunk text, not just a snippet

# # Does the document mention these q4-related terms anywhere? (empty list = no)
# for kw in ["pressure", "unreasonable advantage", "take advantage", "urgent"]:
#     print(kw, "->", find_chunks(kw))  # find_chunks() is defined in the earlier cell

In [4]:

# =============================================================
# 7A.1 BASELINE EVALUATION + SAVE LABELS
# What this does:
#   1. Stores your hand-labeled relevant chunk IDs per query (the "answer key").
#   2. Defines evaluate(), which runs any retrieval function over all eval
#      queries and reports Hit@5 (did a correct chunk appear?) and
#      MRR (how high did the first correct chunk rank?).
#   3. Runs it on the plain FAISS retrieval to get the baseline numbers.
#   4. Saves queries, labels and a text anchor per chunk to eval_set.json,
#      so labels can be re-mapped if chunk IDs change later.
# =============================================================

import json

relevant = {
    "q1": [3, 4],      # three-part unfairness test (split across chunks)
    "q2": [33, 34],    # abusive definition (continues into 34)
    "q3": [19, 27, 53],# fine print can't cure misleading claims / FTC example / disclose fees prominently
    "q4": [33],        # closest match: abusive = unreasonable advantage (inferred, no direct coverage)
    "q5": [18, 22],    # deception test / reasonable consumer perspective
    "q6": [],          # out of scope
}

def evaluate(retrieve_fn, label, k=K):
    print(f"\n--- {label} ---")
    hits, rrs = [], []
    for q in eval_queries:
        results = retrieve_fn(q["query"], k=k)
        ids = [r["chunk_id"] for r in results]
        top = results[0]["score"]
        rel = relevant[q["id"]]
        if rel:
            ranks = [ids.index(c) + 1 for c in rel if c in ids]
            hit = bool(ranks)
            rr = 1 / min(ranks) if ranks else 0.0
            hits.append(hit); rrs.append(rr)
            print(f"{q['id']} {q['type']:<13} hit@{k}={hit!s:<5} rr={rr:.2f} top_score={top:.3f}")
        else:
            print(f"{q['id']} {q['type']:<13} out-of-scope      top_score={top:.3f}")
    print(f"Hit@{k}: {sum(hits)}/{len(hits)} | MRR: {sum(rrs)/len(rrs):.3f}")

evaluate(retrieve, "baseline (FAISS, plain query)")

# Save labels with a text anchor per chunk, so they can be re-mapped if chunk IDs change later
anchors = {cid: chunk_id_to_text[cid][350:410].replace("\n", " ")
           for ids in relevant.values() for cid in ids}
with open("eval_set.json", "w", encoding="utf-8") as f:
    json.dump({"queries": eval_queries, "relevant": relevant, "anchors": anchors}, f, indent=2)


--- baseline (FAISS, plain query) ---
q1 direct        hit@5=True  rr=0.50 top_score=0.781
q2 direct        hit@5=True  rr=0.50 top_score=0.779
q3 vague         hit@5=True  rr=1.00 top_score=0.668
q4 vague         hit@5=False rr=0.00 top_score=0.602
q5 mismatch      hit@5=True  rr=1.00 top_score=0.782
q6 out_of_scope  out-of-scope      top_score=0.541
Hit@5: 4/5 | MRR: 0.600


In [ ]:
# # =============================================================
# # 7A.3 FIX: retry on rate limits (429) as well as server errors
# # What this does: defines a helper that calls Gemini and, if Google
# # says "too many requests" (429) or is overloaded (5xx), waits and
# # tries again instead of crashing.
# # =============================================================
# import time
# from google.genai import errors

# def call_with_retry(fn, retries=5):
#     for attempt in range(retries):
#         try:
#             return fn()
#         except (errors.ClientError, errors.ServerError) as e:
#             code = getattr(e, "code", None)
#             retryable = code in (429, 500, 502, 503, 504)
#             if not retryable or attempt == retries - 1:
#                 raise
#             wait = 15 * (attempt + 1)   # 15s, 30s, 45s, 60s
#             print(f"  (API error {code}, waiting {wait}s then retrying...)")
#             time.sleep(wait)

In [ ]:
# # =============================================================
# # 7A.2 / 7A.3 HELPERS: RETRY + MULTI-QUERY VARIANT GENERATION
# # What this does (defines functions only; makes NO API calls itself):
# #   1. call_with_retry(): wraps any Gemini call; on a rate limit (429)
# #      or server error (5xx) it waits 15s, 30s, ... and retries.
# #      If the error is a PER-DAY quota it stops immediately, because
# #      waiting can't fix that.
# #   2. generate_query_variants(): asks Gemini to rewrite one user
# #      question into 3 differently-worded search queries, using
# #      regulatory vocabulary where it fits.
# # Fixes built into generate_query_variants():
# #   - thinking_budget=0: hidden reasoning tokens count against
# #     max_output_tokens and were truncating the variants
# #   - output cleaning: a regex strips "1."/bullets and a filter drops
# #     heading lines, because Gemini sometimes adds them despite being
# #     told not to
# # Uses gemini-2.5-flash-lite for rewriting (the flash free tier is
# # capped at 20 requests/day).
# # Run order: this cell -> the prep cell -> the 7A.3 retrieval + eval cell.
# # =============================================================
# import re
# import time
# from google.genai import types, errors

# def call_with_retry(fn, retries=5):
#     for attempt in range(retries):
#         try:
#             return fn()
#         except (errors.ClientError, errors.ServerError) as e:
#             code = getattr(e, "code", None)
#             daily_cap = "PerDay" in str(e)   # a per-day quota can't be fixed by waiting
#             retryable = code in (429, 500, 502, 503, 504) and not daily_cap
#             if not retryable or attempt == retries - 1:
#                 raise
#             wait = 15 * (attempt + 1)   # 15s, 30s, 45s, 60s
#             print(f"  (API error {code}, waiting {wait}s then retrying...)")
#             time.sleep(wait)

# def generate_query_variants(query, n=3, model="gemini-2.5-flash-lite"):
#     prompt = f"""You are helping search a CFPB UDAAP (Unfair, Deceptive, or Abusive
# Acts or Practices) examination manual.

# Rewrite the user question below into {n} alternative search queries.
# Rules:
# - Each version should be worded differently from the others and from the original.
# - Use formal regulatory vocabulary where it fits (e.g. "material information",
#   "reasonable consumer", "abusive", "unfair", "deceptive", "disclosure").
# - Keep the meaning of the original question.
# - Return ONLY the {n} queries, one per line, with no numbering or extra text.

# User question: {query}"""
#     config = types.GenerateContentConfig(
#         temperature=0.7,
#         max_output_tokens=1024,
#         thinking_config=types.ThinkingConfig(thinking_budget=0),
#     )
#     response = call_with_retry(lambda: client.models.generate_content(model=model, contents=prompt, config=config))
#     lines = []
#     for l in response.text.strip().split("\n"):
#         l = re.sub(r'^\s*(\d+[.)]|[-•*])\s*', '', l).strip(' "')  # removes "1.", "2)", "-", "•"
#         if l and not l.startswith("<") and not l.endswith(":"):    # drops "<h2>...</h2>" or "Queries:" lines
#             lines.append(l)
#     return lines[:n]

In [8]:
# =============================================================
# HELPERS CELL, EDIT: generate_query_variants for gemini-3.5-flash-lite
# What changed (only this function; call_with_retry stays as is):
#   - model: gemini-2.5-flash-lite is retired for new users, so the
#     model name now lives in REWRITE_MODEL (one place to change it)
#   - thinking: 3.5 models use thinking_level (not thinking_budget);
#     "minimal" is the lowest setting, so reasoning uses few tokens
#   - max_output_tokens raised to 2048 as a safety margin, because
#     thinking tokens still count against the output limit
# =============================================================
REWRITE_MODEL = "gemini-3.5-flash-lite"

def generate_query_variants(query, n=3, model=REWRITE_MODEL):
    prompt = f"""You are helping search a CFPB UDAAP (Unfair, Deceptive, or Abusive
Acts or Practices) examination manual.

Rewrite the user question below into {n} alternative search queries.
Rules:
- Each version should be worded differently from the others and from the original.
- Use formal regulatory vocabulary where it fits (e.g. "material information",
  "reasonable consumer", "abusive", "unfair", "deceptive", "disclosure").
- Keep the meaning of the original question.
- Return ONLY the {n} queries, one per line, with no numbering or extra text.

User question: {query}"""
    config = types.GenerateContentConfig(
        temperature=0.7,
        max_output_tokens=2048,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    )
    response = call_with_retry(lambda: client.models.generate_content(model=model, contents=prompt, config=config))
    lines = []
    for l in response.text.strip().split("\n"):
        l = re.sub(r'^\s*(\d+[.)]|[-•*])\s*', '', l).strip(' "')  # removes "1.", "2)", "-", "•"
        if l and not l.startswith("<") and not l.endswith(":"):    # drops "<h2>...</h2>" or "Queries:" lines
            lines.append(l)
    return lines[:n]

In [ ]:
# # =============================================================
# # DIAGNOSTIC: WHICH GEMINI LIMIT ARE WE HITTING?
# # What this does: makes one tiny generate call and one tiny embed call,
# # and prints Google's error message if either is rejected. The message
# # names the exact quota (per-minute vs per-day) and the model.
# # Nothing is saved or changed.
# # =============================================================
# for label, fn in [
#     ("generate (gemini-2.5-flash)", lambda: client.models.generate_content(model="gemini-2.5-flash", contents="hi")),
#     ("embed (gemini-embedding-001)", lambda: client.models.embed_content(model="gemini-embedding-001", contents="hi")),
# ]:
#     try:
#         fn()
#         print(label, "-> OK")
#     except Exception as e:
#         print(label, "->", str(e)[:700])

generate (gemini-2.5-flash) -> OK
embed (gemini-embedding-001) -> OK


In [9]:
# =============================================================
# 7A.3 PREP: GENERATE ALL VARIANTS ONCE, PACED, SAVED TO DISK
# What this does:
#   1. Loads any variants already saved in query_variants.json
#      (plus anything still in memory from earlier runs).
#   2. Generates variants only for eval queries that don't have them
#      yet, waiting 8s between Gemini calls to stay under the
#      per-minute limit, and saves to disk after EACH query, so a
#      crash never loses finished work.
#   3. Redefines get_variants() so multi_query_retrieve() reads from
#      this cache and makes no more rewrite calls.
# Result: re-running the eval costs only embedding calls, and the
# same variants are used every time, so the numbers are reproducible.
# =============================================================
import json, os, time

VARIANTS_FILE = "query_variants.json"

try:
    variant_cache                      # keep anything already in memory
except NameError:
    variant_cache = {}
if os.path.exists(VARIANTS_FILE):
    with open(VARIANTS_FILE, encoding="utf-8") as f:
        variant_cache.update(json.load(f))

def save_variants():
    with open(VARIANTS_FILE, "w", encoding="utf-8") as f:
        json.dump(variant_cache, f, indent=2)

def get_variants(query):
    if query not in variant_cache:     # only calls Gemini for a query not seen before
        variant_cache[query] = generate_query_variants(query)
        save_variants()
        time.sleep(8)
    return variant_cache[query]

for q in eval_queries:
    already = q["query"] in variant_cache
    variants = get_variants(q["query"])
    print(f"{q['id']} {'(cached)' if already else '(generated)'}: {len(variants)} variants")

save_variants()

q1 (generated): 3 variants
q2 (generated): 3 variants
q3 (generated): 3 variants
q4 (generated): 3 variants
q5 (generated): 3 variants
q6 (generated): 3 variants


In [10]:
# =============================================================
# 7A.3 CHECK: PRINT THE CACHED VARIANTS (NO API CALLS)
# What this does: prints the 3 saved variants for every eval query
# so you can check for junk lines (headings, numbering, cut-off
# text) before they are used for retrieval. It reads variant_cache
# only, so it costs no quota.
# =============================================================
for q in eval_queries:
    print(f"\n{q['id']} original: {q['query']}")
    for i, v in enumerate(variant_cache[q["query"]], 1):
        print(f"   v{i}: {v}")


q1 original: What counts as an unfair act under UDAAP?
   v1: What specific criteria determine whether a practice is unfair under the UDAAP standards
   v2: What constitutes substantial injury and lack of reasonable consumer avoidability for an unfair act
   v3: Legal definition and elements required to establish an unfair practice in regulatory examinations

q2 original: How does UDAAP define an abusive practice?
   v1: What are the statutory criteria for establishing an abusive act or practice under the CFPB examination manual?
   v2: How does the regulatory framework determine when a financial practice meets the threshold of being abusive?
   v3: What specific elements must be present to classify conduct as abusive according to UDAAP standards?

q3 original: Can we hide fees in the fine print?
   v1: Is it considered a deceptive practice to conceal material fee disclosures within fine print?
   v2: What are the regulatory standards regarding the prominent display of fees versus bur

In [11]:
# =============================================================
# 7A.3 MULTI-QUERY RETRIEVAL + RECIPROCAL RANK FUSION (RRF)
# What this does:
#   1. Takes the original query plus its 3 saved variants (from the
#      prep cell's cache, so no new rewrite calls) = 4 searches.
#   2. Searches FAISS with each of the 4 queries (top 10 each).
#   3. Merges the 4 ranked lists with RRF: each chunk scores
#      sum(1 / (60 + rank)) over the lists it appears in, so chunks
#      that several variants agree on rise to the top.
#   4. Returns the top-k merged chunks in the same dict format as
#      retrieve(), so evaluate() works on it unchanged.
#   5. Runs the same eval as the baseline so you can compare.
# Requires: the prep cell (variants cache + get_variants) run first.
# Note: the "score" returned here is an RRF score (max about 0.066),
# NOT cosine similarity. Don't compare top_score with the baseline.
# =============================================================
from collections import defaultdict

def multi_query_retrieve(query, k=K, per_query_k=10, rrf_k=60):
    all_queries = [query] + get_variants(query)       # original + 3 variants
    fused = defaultdict(float)                         # chunk_id -> RRF score
    texts = {}
    for q in all_queries:
        for rank, r in enumerate(retrieve(q, k=per_query_k), start=1):
            fused[r["chunk_id"]] += 1 / (rrf_k + rank)
            texts[r["chunk_id"]] = r["text"]
    top = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]
    return [{"chunk_id": cid, "score": s, "text": texts[cid]} for cid, s in top]

# Run the same evaluation as the baseline, with multi-query retrieval
evaluate(multi_query_retrieve, "multi-query + RRF")

# Show what multi-query retrieved for q4 (the baseline miss)
print("\nq4 top-5 with multi-query:")
for r in multi_query_retrieve(eval_queries[3]["query"]):
    print(f"  chunk {r['chunk_id']:>2} | rrf={r['score']:.4f} | {r['text'][:110].replace(chr(10), ' ')}...")


--- multi-query + RRF ---
q1 direct        hit@5=True  rr=1.00 top_score=0.063
q2 direct        hit@5=True  rr=0.50 top_score=0.064
q3 vague         hit@5=True  rr=0.50 top_score=0.063
q4 vague         hit@5=False rr=0.00 top_score=0.046
q5 mismatch      hit@5=True  rr=0.50 top_score=0.064
q6 out_of_scope  out-of-scope      top_score=0.065
Hit@5: 4/5 | MRR: 0.500

q4 top-5 with multi-query:
  chunk 20 | rrf=0.0460 | bsequent truthful disclosures.  Acts or practices that may be deceptive include: making misleading cost or pri...
  chunk  2 | rrf=0.0448 | otential harm to the consumer associated with the product.    These examination procedures provide general gui...
  chunk 22 | rrf=0.0320 | ce is misleading, one also must consider whether the  consumer’s interpretation of or reaction to the represen...
  chunk  0 | rrf=0.0313 | CFPB Consumer  Laws and Regulations UDAAP  CFPB Manual V.2 (October 2012) UDAAP 1  Unfair, Deceptive, or Abusi...
  chunk 47 | rrf=0.0313 | equate to prevent u